In [1]:
!uv add pandas

Resolved 40 packages in 26.59s
Prepared 2 packages in 2m 01s
Installed 4 packages in 587ms
 + numpy==2.3.3
 + pandas==2.3.2
 + pytz==2025.2
 + tzdata==2025.2


In [1]:
from concurrent.futures import ProcessPoolExecutor as ppe
from generate import generate_csv

NUMBER_OF_FILES = 5
NUMBER_OF_LINES = 1000

if __name__ == "__main__":
    with ppe() as executor:
        futures = [executor.submit(generate_csv,i,NUMBER_OF_LINES) for i in range(1,NUMBER_OF_FILES + 1)]

fn = []
for future in futures:
    fn.append(future.result())
    
futures

[<Future at 0x2506c1f87d0 state=finished returned str>,
 <Future at 0x2506c1f8a50 state=finished returned str>,
 <Future at 0x2506c1669e0 state=finished returned str>,
 <Future at 0x2506c166c40 state=finished returned str>,
 <Future at 0x2506c18e330 state=finished returned str>]

In [2]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor as tpe

def process(path):
    df = pd.read_csv(path)
    return df.groupby('Категория')['Значение'].agg(Медиана = 'median', Стандартное_отклонение='std').reset_index()

with tpe() as executor:
    dataframes = list(executor.map(process,fn))

combined = pd.concat(dataframes, ignore_index=True)

combined

,Категория,Медиана,Стандартное_отклонение
0,A,5057.682826,3021.652151
1,B,5602.589517,2913.453126
2,C,4613.772569,2938.548954
3,D,4626.787232,2974.424803
4,A,5058.561509,2654.980190
5,B,5084.369905,2874.320122
6,C,5094.174036,2771.712780
7,D,4702.901937,2829.880289
8,A,4714.790565,2934.960064
9,B,4672.043640,2877.342924


In [3]:
result = combined.groupby('Категория')['Медиана'].agg(Медиана_из_медиан='median', Стандартное_отклонение_из_медиан='std').reset_index()
result

,Категория,Медиана_из_медиан,Стандартное_отклонение_из_медиан
0,A,5057.682826,151.592070
1,B,5084.369905,340.854029
2,C,4824.550545,219.544191
3,D,4739.167011,267.756799
